# Dokumentasi Konfigurasi Jaringan
## SI069764_INTERNET_100M_SINGLE_KALBE_FARMA_CIKARANG

---

## 1. Informasi Layanan

| Item | Keterangan |
|---|---|
| Site / Customer | Kalbe Farma Cikarang |
| Service ID | SI069764 |
| Layanan | Internet 100 Mbps Single |
| Router | MikroTik iForte RB1100AHx4 |
| Lokasi Radio / POE | Lantai 5 - Rooftop |
| Lokasi Router / Server | Lantai 1 - Ruang Server |
| Time Zone | Asia/Jakarta |

---

## 2. Topologi Jaringan

```mermaid
flowchart TD
    A[Radio Client<br/>Lantai 5 Rooftop]
    B[Panel POE & Listrik]
    C[Converter]
    D[HTB]
    E[MikroTik iForte<br/>RB1100AHx4]
    F[Perangkat Kalbe Farma]

    A --> B
    B --> C
    C -->|Patchcore| D
    D -->|LAN / WAN Wireless| E
    E -->|ether2 / Public IP| F
    E -->|ether3-ether4 / LAN Private| F
```

### Penjelasan Topologi

| Perangkat | Fungsi | Lokasi |
|---|---|---|
| Radio Client | Menerima koneksi wireless dari provider | Lantai 5 / Rooftop |
| Panel POE & Listrik | Menyediakan power ke radio client | Lantai 5 / Rooftop |
| Converter | Konversi media koneksi dari sisi rooftop | Jalur menuju ruang server |
| HTB | Terminasi koneksi sebelum masuk router | Ruang Server |
| MikroTik iForte RB1100AHx4 | Router utama untuk WAN, Public IP, LAN, NAT, DHCP, DNS, SNMP | Lantai 1 / Ruang Server |
| Perangkat Kalbe Farma | Perangkat LAN/customer yang menerima koneksi internet | Lantai 1 / Ruang Server |

---

## 3. Mapping Interface MikroTik

| Interface | Fungsi | Keterangan |
|---|---|---|
| `ether1` | WAN Wireless | Interface fisik untuk VLAN WAN |
| `WAN Wireless` | VLAN WAN | VLAN ID `2455` di atas `ether1` |
| `ether2` | Public IP LAN | Digunakan untuk IP publik customer |
| `ether3` | LAN Private | Masuk ke bridge private |
| `ether4` | LAN Private | Masuk ke bridge private |
| `Bridge-LAN_Private` | Bridge LAN private | Bridge untuk jaringan private `192.168.100.0/24` |

---

## 4. Konfigurasi VLAN WAN

### Interface VLAN

| Parameter | Nilai |
|---|---|
| Name | `WAN Wireless` |
| Interface Parent | `ether1` |
| VLAN ID | `2455` |

### Penjelasan

VLAN `2455` dibuat di atas interface `ether1` dan digunakan sebagai jalur WAN wireless dari sisi provider menuju router MikroTik.

---

## 5. IP Addressing

### IP WAN

| Parameter | Nilai |
|---|---|
| Address | `100.70.161.142/30` |
| Interface | `WAN Wireless` |
| Gateway WAN | `100.70.161.141` |

### IP Public Customer

| Parameter | Nilai |
|---|---|
| Address | `103.165.123.185/29` |
| Interface | `ether2` |
| Subnet Mask | `255.255.255.248` |
| IP Available | `103.165.123.186 - 103.165.123.190` |
| Gateway Customer | `103.165.123.185` |
| Port Customer | `ether2` |

### IP LAN Private

| Parameter | Nilai |
|---|---|
| Address | `192.168.100.1/24` |
| Interface | `Bridge-LAN_Private` |
| Network | `192.168.100.0/24` |

> Catatan: Karena `ether3` dimasukkan ke `Bridge-LAN_Private`, IP LAN private idealnya dipasang pada interface bridge, bukan langsung di `ether3`.

---

## 6. Static Route

| Parameter | Nilai |
|---|---|
| Dst. Address | `0.0.0.0/0` |
| Gateway | `100.70.161.141` |

### Penjelasan

Default route diarahkan ke gateway WAN provider, yaitu `100.70.161.141`, agar seluruh trafik internet keluar melalui jalur WAN Wireless.

---

## 7. DNS

| Parameter | Nilai |
|---|---|
| Primary DNS | `202.51.96.7` |
| Secondary DNS | `202.51.102.118` |
| Allow Remote Request | Tidak dicentang / disabled |

### Penjelasan

Router menggunakan DNS dari provider. Opsi `Allow Remote Request` tidak diaktifkan agar router tidak menjadi DNS resolver terbuka.

---

## 8. NAT Firewall

### Rule NAT

| Parameter | Nilai |
|---|---|
| Chain | `srcnat` |
| Src. Address | `192.168.100.0/24` |
| Action | `src-nat` |
| To Address | `103.165.123.185` |

### Penjelasan

Jaringan private `192.168.100.0/24` akan diterjemahkan menggunakan IP publik `103.165.123.185` saat mengakses internet.

---

## 9. Bridge LAN Private

### Bridge

| Parameter | Nilai |
|---|---|
| Name | `Bridge-LAN_Private` |
| STP | `none` |

### Bridge Port

| Interface | Bridge |
|---|---|
| `ether3` | `Bridge-LAN_Private` |
| `ether4` | `Bridge-LAN_Private` |

### Penjelasan

`ether3` dan `ether4` digabungkan dalam satu bridge untuk jaringan LAN private customer.

---

## 10. IP Service

### Service yang Diaktifkan

| Service | Port | Status |
|---|---:|---|
| Telnet | `27292` | Enabled |
| Winbox | `36459` | Enabled |

### Service yang Dinonaktifkan

| Service | Status |
|---|---|
| FTP | Disabled |
| SSH | Disabled |
| WWW | Disabled |
| API | Disabled |
| API-SSL | Disabled |
| Service lain yang tidak digunakan | Disabled |

### Penjelasan

Akses manajemen router hanya dibuka melalui Telnet dan Winbox dengan port custom. Service lain dinonaktifkan untuk mengurangi risiko akses tidak perlu.

---

## 11. DHCP Server

### DHCP Setup

| Parameter | Nilai |
|---|---|
| DHCP Server Interface | `Bridge-LAN_Private` |
| DHCP Address Space | `192.168.100.0/24` |
| Gateway for DHCP Network | `192.168.100.1` |
| Address to Give Out | `192.168.100.2 - 192.168.100.250` |
| DNS Server | `202.51.96.7`, `202.51.102.118` |
| Lease Time | Default / sesuai kebutuhan |

### Penjelasan

DHCP Server berjalan di `Bridge-LAN_Private` dan memberikan IP otomatis untuk perangkat LAN private pada subnet `192.168.100.0/24`.

---

## 12. SNMP

### SNMP Setting

| Parameter | Nilai |
|---|---|
| SNMP | Enabled |
| Community Name | `durentiga` |
| Community Address | `0.0.0.0/0` |
| Trap Community | `durentiga` |
| Trap Version | `2` |

### Penjelasan

SNMP diaktifkan untuk kebutuhan monitoring perangkat MikroTik. Community yang digunakan adalah `durentiga`.

---

## 13. System Identity & Time

| Parameter | Nilai |
|---|---|
| Identity | `SI069764_INTERNET_100M_SINGLE_KALBE_FARMA_CIKARANG` |
| Time | Disesuaikan |
| Date | Disesuaikan |
| Time Zone Name | `Asia/Jakarta` |

---

## 14. Ringkasan IP Publik

| Item | Nilai |
|---|---|
| Network | `103.165.123.184/29` |
| Gateway | `103.165.123.185` |
| Usable IP | `103.165.123.186 - 103.165.123.190` |
| Broadcast | `103.165.123.191` |
| Subnet Mask | `255.255.255.248` |
| DNS 1 | `202.51.96.7` |
| DNS 2 | `202.51.102.118` |
| Customer Port | `ether2` |

---

## 15. Ringkasan Alur Koneksi

1. Koneksi provider diterima oleh `Radio Client` di rooftop.
2. Radio mendapat power dari `Panel POE & Listrik`.
3. Jalur turun menuju `Converter`.
4. Dari converter masuk ke `HTB` melalui patchcore.
5. Dari HTB masuk ke MikroTik iForte RB1100AHx4.
6. MikroTik menerima WAN melalui VLAN `2455` pada `ether1`.
7. Public IP customer tersedia pada `ether2`.
8. LAN private customer berjalan pada bridge `Bridge-LAN_Private` melalui `ether3` dan `ether4`.
9. Trafik LAN private `192.168.100.0/24` keluar internet menggunakan NAT ke `103.165.123.185`.
